In [1]:
# 0. Install

#!pip install openai

In [2]:
# 1. Establish API key

import os

#GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

#ANTHROPIC_API_KEY

if OPENAI_API_KEY is None:
    raise RuntimeError("OPENAI_API_KEY not found in environment.")

In [3]:
# 2. Set automation constants
import os, json

FEATURES = [
    "AktAcc", "AktAch", "AktAct", "AktSemel", "AktState",
    "ASTrns", "ASUncc", "ASUnrg", "ClCntfcl", "ClNegPol",
    "DtDTAM", "DtExp", "DtRes", "DtUniv",
    "TLAspPfv", "TLMdIrr", "TLTS", "VcPass",
]

PERMUTATIONS = [
    {"type": "full",     "component": None},
    {"type": "ablation", "component": "intuition"},
    {"type": "ablation", "component": "set_to_1_if"},
    {"type": "ablation", "component": "set_to_0_if"},
    {"type": "ablation", "component": "edge_cases"},
    {"type": "ablation", "component": "examples_positive"},
    {"type": "ablation", "component": "examples_negative"},
   # {"type": "allation", "component": "intuition"},
   # {"type": "allation", "component": "set_to_1_if"},
   # {"type": "allation", "component": "set_to_0_if"},
   # {"type": "allation", "component": "edge_cases"},
   # {"type": "allation", "component": "examples_positive"},
   # {"type": "allation", "component": "examples_negative"},
]

BASE_ROOT  = "/Users/glossophilia/Desktop/cri_runs/OpenAI"
CRI_DIR    = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/CRIs"
INPUT_CSV  = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Input/ntb_input.csv"
CHECKPOINT = os.path.join(BASE_ROOT, "completed_runs.json")

In [4]:
# 3. Run config specs

run_config = {
  "run_id": None,

  #"feature_name": "DtUniv",  # <-- set this

  "model_provider": "openai",  # <-- set this
  "model_name": "gpt-4.1-mini-2025-04-14",  # <-- set this
    # Set input_csv_path to match data input file and cri_json_path to feature-specific CRI file
  "input_csv_path": "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Input/ntb_input.csv",
  #"cri_json_path": "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/CRIs/DtUniv.json",

  "data": {
      "max_tokens": 350,    # None = all; total tokens to draw from dataset
      "batch_size": 50,     # tokens per output file
      "start_index": 0,
      "shuffle": False,
      "random_seed": 0
  },

 # "permutation": {
 #   "type": "ablation",            # full | ablation | allation
 #   "component": "intuition"
 # },

  "prompt": {
    "version": "v1",
    "system_template_hash": None,
    "assembled_prompt_hash": None
  },

  "inference": {
    "temperature": 0.0,
    "max_tokens": 512,
    "seed": None,
    "rate_limit_sleep_s": 0.1,
    "max_retries": 5
  },

  "io": {
    "output_dir": None,
    "jsonl_output_path": None,
    "csv_output_path": None
  },

  "notes": ""
}

def _derive_cri_variant(permutation: dict) -> str:
    abbrev   = {"full": "full", "ablation": "abl", "allation": "all"}
    comp_idx = {
        "intuition": 1, "set_to_1_if": 2, "set_to_0_if": 3,
        "edge_cases": 4, "examples_positive": 5, "examples_negative": 6,
    }
    t = abbrev.get(permutation["type"], permutation["type"])
    c = permutation.get("component")
    return t if c is None else f"{t}{comp_idx.get(c, c)}"

#run_config["cri_variant"] = _derive_cri_variant(run_config["permutation"])
#print("cri_variant:", run_config["cri_variant"])

In [5]:
# 3. Set checkpoints

def _ckpt_key(feature_name, permutation):
    return f"{feature_name}::{_derive_cri_variant(permutation)}"

def load_checkpoint():
    if os.path.exists(CHECKPOINT):
        with open(CHECKPOINT) as f:
            return set(json.load(f))
    return set()

def save_checkpoint(completed):
    with open(CHECKPOINT, "w") as f:
        json.dump(sorted(completed), f, indent=2)

In [6]:
# 4. Load CRI json file and implement permutation (full/ablation/allation)

import json
from copy import deepcopy

CRI_COMPONENTS = [
    "intuition",
    "set_to_1_if",
    "set_to_0_if",
    "edge_cases",
    "examples_positive",
    "examples_negative",
]

def load_cri(cri_json_path: str) -> dict:
    with open(cri_json_path, "r", encoding="utf-8") as f:
        cri = json.load(f)
    return cri

def apply_permutation(cri: dict, permutation: dict) -> dict:
    """
    Returns a CRI dict containing ONLY the active components for this run.
    - full: all components kept
    - ablation: drop exactly one named component
    - allation: keep exactly one named component
    """
    ptype = permutation["type"]
    component = permutation.get("component")

    # Defensive copy
    cri_active = deepcopy(cri)

    if ptype == "full":
        # Ensure we only keep the known components (avoids stray metadata leaking into prompt)
        return {k: cri_active.get(k) for k in CRI_COMPONENTS if k in cri_active}

    if component not in CRI_COMPONENTS:
        raise ValueError(f"Invalid component '{component}'. Must be one of: {CRI_COMPONENTS}")

    if ptype == "ablation":
        return {k: cri_active.get(k) for k in CRI_COMPONENTS if k != component and k in cri_active}

    if ptype == "allation":
        return {component: cri_active.get(component)}

    raise ValueError(f"Invalid permutation type '{ptype}'. Must be: full | ablation | allation")

# ---- Fill these in for your current run ----
# Set desired output folder path
#run_config["cri_json_path"] = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/cris/DtExp.json"   # <-- set this
# run_config["permutation"] = {"type": "ablation", "component": "edge_cases"}  # example
# run_config["permutation"] = {"type": "allation", "component": "set_to_1_if"} # example

#cri = load_cri(run_config["cri_json_path"])
#cri_active = apply_permutation(cri, run_config["permutation"])

#print("Active CRI components:", list(cri_active.keys()))

In [7]:
# 5. Assemble a deterministic prompt string + write its hash into run_config

import hashlib

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

# META-INSTRUCTIONS block (commented out — not included in active template)
# META_SECTION = """\
# META-INSTRUCTIONS (how to use the CRIs):
# {meta_block}
#
# """

PROMPT_TEMPLATE_V1 = """\
You are a trained linguist performing binary semantic feature annotation.

TASK:
Decide whether the TARGET GRAM instantiates the feature: {feature_name}.

EVIDENCE RULES:
- Use only the TARGET GRAM and the FULL TEXT provided.
- Follow the Constrained Reasoning Instructions (CRIs) exactly as written.
- Do not assume missing context or world knowledge beyond what is in the FULL TEXT.

CRIs:
{cri_block}

INPUT:
TokenID: {TokenID}
Target: {Target}
FullText: {FullText}

OUTPUT (STRICT JSON ONLY; no extra text):
{{
  "TokenID": "{TokenID}",
  "annotation": 0 or 1,
  "confidence": "high" or "medium" or "low",
  "explanation": "one sentence"
}}
"""

def render_meta_block(cri_active: dict) -> str:
    lines = []

    # Always present
    lines.append("Treat each CRI component as a different kind of evidence. Follow them as written.")

    # Only if set_to_1_if exists
    if "set_to_1_if" in cri_active:
        lines.append("Assign 1 if at least one 'set_to_1_if' condition applies.")

    # Only if set_to_0_if exists
    if "set_to_0_if" in cri_active:
        lines.append(
            "If any 'set_to_0_if' condition applies, assign 0 (this vetoes 'set_to_1_if'), "
            "unless an 'edge_cases' instruction explicitly overrides."
        )

    return "- " + "\n- ".join(lines)

def render_cri_block(cri_active: dict) -> str:
    parts = []
    for key in CRI_COMPONENTS:
        if key in cri_active and cri_active[key] is not None:
            parts.append(f"[{key}]\n{cri_active[key]}")
    return "\n\n".join(parts)

def build_prompt(feature_name: str, cri_active: dict, token: dict) -> str:
    cri_block = render_cri_block(cri_active)
    # meta_block = render_meta_block(cri_active)

    return PROMPT_TEMPLATE_V1.format(
        feature_name=feature_name,
        cri_block=cri_block,
        # meta_block=meta_block,
        TokenID=token["TokenID"],
        Target=token["Target"],
        FullText=token["FullText"],
    )

# ---- store template hash in config ----
run_config["prompt"]["version"] = "v1"
run_config["prompt"]["system_template_hash"] = sha256_text(PROMPT_TEMPLATE_V1)

print("Template hash:", run_config["prompt"]["system_template_hash"])

Template hash: 3037f3175824ebb7a8ba75c133cebfab094b661cf2ccc228c623ed63e4eaecbd


In [8]:
# 6. Load input CSV, apply slice, set output filenames, and render one dry-run prompt

import pandas as pd
import os
import json
from datetime import datetime

# ---- Load CSV once ----
df = pd.read_csv(run_config["input_csv_path"])

required_columns = {"TokenID", "Target", "FullText"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ---- Apply run_config data slicing deterministically ----
data_cfg = run_config["data"]

df_work = df
if data_cfg.get("shuffle", False):
    df_work = df_work.sample(frac=1, random_state=data_cfg.get("random_seed", 0)).reset_index(drop=True)

start = int(data_cfg.get("start_index", 0))
max_tokens = data_cfg.get("max_tokens", None)

df_slice = df_work.iloc[start:] if max_tokens is None else df_work.iloc[start:start + int(max_tokens)]
tokens = df_slice.to_dict(orient="records")

print(f"Total tokens in file: {len(df)}")
print(f"Tokens selected for this run: {len(tokens)}")

if not tokens:
    raise RuntimeError("No tokens selected for this run (tokens list is empty).")

Total tokens in file: 349
Tokens selected for this run: 349


In [9]:
# 7. Strict response parser + validator

import json
import re

_FENCE_RE = re.compile(r"^\s*```(?:json)?\s*\n([\s\S]*?)\n```\s*$", re.IGNORECASE)

def parse_model_response(raw_text: str) -> dict:
    """
    Parses model output and enforces strict JSON schema.
    Accepts either:
      - pure JSON
      - JSON wrapped in a single ```json ... ``` fence
    """
    text = raw_text.strip()

    m = _FENCE_RE.match(text)
    if m:
        text = m.group(1).strip()

    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model output is not valid JSON. JSONDecodeError: {e}")

    required_keys = {"TokenID", "annotation", "confidence", "explanation"}
    if set(data.keys()) != required_keys:
        raise ValueError(f"JSON keys mismatch. Expected {required_keys}, got {set(data.keys())}")

    if data["annotation"] not in (0, 1):
        raise ValueError("Annotation must be 0 or 1.")

    if data["confidence"] not in ("high", "medium", "low"):
        raise ValueError("Confidence must be 'high', 'medium', or 'low'.")

    if not isinstance(data["TokenID"], str):
        raise ValueError("TokenID must be a string.")

    if not isinstance(data["explanation"], str):
        raise ValueError("Explanation must be a string.")

    return data

In [10]:
# 8. Model call wrapper (OpenAI + provider dispatcher)

import time
import openai
from openai import OpenAI, RateLimitError, APIError, APITimeoutError, APIConnectionError

# ---- OpenAI client (60s request timeout) ----
client = OpenAI(api_key=OPENAI_API_KEY, timeout=60.0)

def call_openai(prompt: str) -> str:
    max_retries = int(run_config["inference"].get("max_retries", 3))
    base_sleep  = float(run_config["inference"].get("rate_limit_sleep_s", 1.0))
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=run_config["model_name"],
                messages=[{"role": "user", "content": prompt}],
                temperature=run_config["inference"]["temperature"],
                max_completion_tokens=int(run_config["inference"]["max_tokens"]),
            )
            return response.choices[0].message.content
        except RateLimitError as e:
            last_err = e
            sleep_s = max(base_sleep, 1.0) * (2 ** (attempt - 1))
            print(f"[429] Rate limited. Sleeping {sleep_s:.1f}s then retrying ({attempt}/{max_retries})...")
            time.sleep(sleep_s)
            continue
        except (APITimeoutError, APIConnectionError) as e:
            last_err = e
            sleep_s = max(base_sleep, 1.0) * (2 ** (attempt - 1))
            print(f"[TIMEOUT/CONN] {type(e).__name__}. Sleeping {sleep_s:.1f}s then retrying ({attempt}/{max_retries})...")
            time.sleep(sleep_s)
            continue
        except APIError as e:
            raise
    raise RuntimeError(f"OpenAI call failed after {max_retries} retries. Last error: {last_err}")

# ---- Provider dispatcher ----
def call_model(prompt: str) -> str:
    provider = run_config["model_provider"]
    if provider == "openai":
        return call_openai(prompt)
    else:
        raise ValueError(f"Unknown model_provider: {provider}")

In [11]:
# Single-token live test

#first_token = tokens[0]

#prompt0 = build_prompt(
#    run_config["feature_name"],
#    cri_active,
#    first_token
#)

#print("Sending prompt for TokenID:", first_token["TokenID"])

#raw_output = call_model(prompt0)

#print("\n--- RAW MODEL OUTPUT ---\n")
#print(raw_output)
#print("\n--- END RAW OUTPUT ---\n")

In [ ]:
# 9. Automated full run — all FEATURES × PERMUTATIONS
import json, math, os, time, uuid
import pandas as pd
from datetime import datetime

# SMOKE TEST — remove these three lines before the real run
#FEATURES     = ["DtUniv"]
#PERMUTATIONS = [{"type": "full", "component": None}]
#run_config["data"]["max_tokens"] = 2

completed = load_checkpoint()
print(f"Resuming: {len(completed)} runs already completed.\n")

for feature_name in FEATURES:
    cri_json_path = os.path.join(CRI_DIR, f"{feature_name}.json")

    for permutation in PERMUTATIONS:
        ck = _ckpt_key(feature_name, permutation)
        if ck in completed:
            print(f"[SKIP] {ck}")
            continue

        cri_variant    = _derive_cri_variant(permutation)
        timestamp      = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
        run_id         = f"{timestamp}_{str(uuid.uuid4())[:8]}"
        run_output_dir = os.path.join(BASE_ROOT, feature_name, run_id)
        os.makedirs(run_output_dir, exist_ok=True)

        # --- Update run_config in-place so call_model() keeps working ---
        run_config["run_id"]                  = run_id
        run_config["feature_name"]            = feature_name
        run_config["cri_json_path"]           = cri_json_path
        run_config["input_csv_path"]          = INPUT_CSV
        run_config["permutation"]             = permutation
        run_config["cri_variant"]             = cri_variant
        run_config["io"]["output_dir"]        = run_output_dir
        run_config["io"]["jsonl_output_path"] = None
        run_config["io"]["csv_output_path"]   = None

        config_path = os.path.join(run_output_dir, "run_config.json")
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(run_config, f, indent=2)

        try:
            # --- Load CRI ---
            cri        = load_cri(cri_json_path)
            cri_active = apply_permutation(cri, permutation)
            print(f"\n{'='*60}")
            print(f"STARTING : {feature_name} | {cri_variant}")
            print(f"run_id   : {run_id}")
            print(f"Active CRI: {list(cri_active.keys())}")
            print(f"{'='*60}")

            # --- Slice data pool (same logic as Cell 8) ---
            data_cfg   = run_config["data"]
            df_work    = df
            if data_cfg.get("shuffle"):
                df_work = df_work.sample(
                    frac=1, random_state=data_cfg.get("random_seed", 0)
                ).reset_index(drop=True)
            start_idx = int(data_cfg.get("start_index", 0))
            max_tok   = data_cfg.get("max_tokens")
            df_pool   = (
                df_work.iloc[start_idx:] if max_tok is None
                else df_work.iloc[start_idx : start_idx + int(max_tok)]
            )
            batch_size = int(data_cfg.get("batch_size", 50))
            n_batches  = math.ceil(len(df_pool) / batch_size)
            print(f"Batches: {n_batches} × {batch_size} tokens = {len(df_pool)} total\n")

            # --- Batch loop ---
            jsonl_paths = []
            for batch_idx in range(n_batches):
                run_tokens = df_pool.iloc[
                    batch_idx * batch_size : (batch_idx + 1) * batch_size
                ].to_dict(orient="records")

                first_tok  = str(run_tokens[0]["TokenID"])
                last_tok   = str(run_tokens[-1]["TokenID"])
                date_str   = datetime.now().strftime("%Y%m%d")
                base       = f"{feature_name}_{run_config['model_provider']}_{cri_variant}_{first_tok}_{last_tok}_{date_str}"
                jsonl_path = os.path.join(run_output_dir, f"{base}.jsonl")

                run_config["io"]["jsonl_output_path"] = jsonl_path
                run_config["io"]["csv_output_path"]   = os.path.join(run_output_dir, f"{base}.csv")
                jsonl_paths.append(jsonl_path)

                print(f"\n{'='*60}")
                print(f"RUN : {feature_name} | {run_config['model_provider']} | {cri_variant}  [batch {batch_idx+1}/{n_batches}]")
                print(f"N   : {len(run_tokens)} tokens  ({first_tok} → {last_tok})")
                print(f"OUT : {base}.jsonl")
                print(f"{'='*60}\n")

                processed = 0
                parse_failures = 0
                with open(jsonl_path, "w", encoding="utf-8") as out_f:
                    for token in run_tokens:
                        token_id = token["TokenID"]
                        prompt   = build_prompt(feature_name, cri_active, token)
                        try:
                            raw    = call_model(prompt)
                            parsed = parse_model_response(raw)
                            parsed["run_id"]       = run_config["run_id"]
                            parsed["provider"]     = run_config["model_provider"]
                            parsed["model_name"]   = run_config["model_name"]
                            parsed["cri_variant"]  = cri_variant
                            parsed["feature_name"] = feature_name
                            out_f.write(json.dumps(parsed) + "\n")
                            processed += 1
                        except Exception as e:
                            out_f.write(json.dumps({
                                "TokenID":      token_id,
                                "annotation":   None,
                                "confidence":   None,
                                "explanation":  None,
                                "error":        str(e),
                                "cri_variant":  cri_variant,
                                "feature_name": feature_name,
                            }) + "\n")
                            parse_failures += 1
                        time.sleep(run_config["inference"]["rate_limit_sleep_s"])

                print(f"Done: {processed} processed, {parse_failures} parse failures")

            # --- Merge all batch JSONL → single CSV (Cell 9 logic, inlined) ---
            records = []
            for path in jsonl_paths:
                with open(path, "r", encoding="utf-8") as f:
                    for line in f:
                        records.append(json.loads(line))
            df_results = pd.DataFrame(records).drop(
                columns=["run_id", "provider", "model_name", "cri_variant", "feature_id"],
                errors="ignore",
            )
            first_tok_full  = str(df_pool.iloc[0]["TokenID"])
            last_tok_full   = str(df_pool.iloc[-1]["TokenID"])
            date_str        = datetime.now().strftime("%Y%m%d")
            csv_output_path = os.path.join(
                run_output_dir,
                f"{feature_name}_{run_config['model_provider']}_{cri_variant}"
                f"_{first_tok_full}_{last_tok_full}_{date_str}.csv",
            )
            df_meta = df_pool.copy()
            df_meta["run_id"]       = run_config["run_id"]
            df_meta["provider"]     = run_config["model_provider"]
            df_meta["model_name"]   = run_config["model_name"]
            df_meta["feature_name"] = feature_name
            df_merged = df_meta.merge(df_results, on="TokenID", how="left")
            df_merged.to_csv(csv_output_path, index=False)
            print(f"Merged {len(records)} records → {csv_output_path}")

            # --- Checkpoint ---
            completed.add(ck)
            save_checkpoint(completed)
            print(f"[CHECKPOINT] {ck}")

        except Exception as e:
            print(f"\n[ERROR] {ck} failed: {e}")
            print("Skipping to next run.\n")

print("\n=== ALL AUTOMATION RUNS COMPLETE ===")

Resuming: 33 runs already completed.

[SKIP] AktAcc::full
[SKIP] AktAcc::abl1
[SKIP] AktAcc::abl2
[SKIP] AktAcc::abl3
[SKIP] AktAcc::abl4
[SKIP] AktAcc::abl5
[SKIP] AktAcc::abl6
[SKIP] AktAch::full
[SKIP] AktAch::abl1
[SKIP] AktAch::abl2
[SKIP] AktAch::abl3
[SKIP] AktAch::abl4
[SKIP] AktAch::abl5
[SKIP] AktAch::abl6
[SKIP] AktAct::full
[SKIP] AktAct::abl1
[SKIP] AktAct::abl2
[SKIP] AktAct::abl3
[SKIP] AktAct::abl4
[SKIP] AktAct::abl5
[SKIP] AktAct::abl6
[SKIP] AktSemel::full
[SKIP] AktSemel::abl1
[SKIP] AktSemel::abl2
[SKIP] AktSemel::abl3
[SKIP] AktSemel::abl4
[SKIP] AktSemel::abl5
[SKIP] AktSemel::abl6
[SKIP] AktState::full
[SKIP] AktState::abl1
[SKIP] AktState::abl2
[SKIP] AktState::abl3
[SKIP] AktState::abl4

STARTING : AktState | abl5
run_id   : 20260325T153847Z_4342e174
Active CRI: ['intuition', 'set_to_1_if', 'set_to_0_if', 'edge_cases', 'examples_negative']
Batches: 7 × 50 tokens = 349 total


RUN : AktState | openai | abl5  [batch 1/7]
N   : 50 tokens  (40001018 → 40019027)
OU

/var/folders/22/4lgkjkkn2vvb2m8ffgsjl7gh0000gn/T/ipykernel_46633/856294939.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp      = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl5  [batch 2/7]
N   : 50 tokens  (40019028 → 41004029)
OUT : AktState_openai_abl5_40019028_41004029_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl5  [batch 3/7]
N   : 50 tokens  (41005014 → 42001001a)
OUT : AktState_openai_abl5_41005014_42001001a_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl5  [batch 4/7]
N   : 50 tokens  (42001001b → 42008056)
OUT : AktState_openai_abl5_42001001b_42008056_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl5  [batch 5/7]
N   : 50 tokens  (42009008a → 42019037)
OUT : AktState_openai_abl5_42009008a_42019037_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl5  [batch 6/7]
N   : 50 tokens  (42019046 → 43005033)
OUT : AktState_openai_abl5_42019046_43005033_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl5  [batch 7/7]
N 

/var/folders/22/4lgkjkkn2vvb2m8ffgsjl7gh0000gn/T/ipykernel_46633/856294939.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp      = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl6  [batch 2/7]
N   : 50 tokens  (40019028 → 41004029)
OUT : AktState_openai_abl6_40019028_41004029_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl6  [batch 3/7]
N   : 50 tokens  (41005014 → 42001001a)
OUT : AktState_openai_abl6_41005014_42001001a_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl6  [batch 4/7]
N   : 50 tokens  (42001001b → 42008056)
OUT : AktState_openai_abl6_42001001b_42008056_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl6  [batch 5/7]
N   : 50 tokens  (42009008a → 42019037)
OUT : AktState_openai_abl6_42009008a_42019037_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl6  [batch 6/7]
N   : 50 tokens  (42019046 → 43005033)
OUT : AktState_openai_abl6_42019046_43005033_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : AktState | openai | abl6  [batch 7/7]
N 

...

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl1  [batch 2/7]
N   : 50 tokens  (40019028 → 41004029)
OUT : ClCntfcl_openai_abl1_40019028_41004029_20260325.jsonl

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl1  [batch 3/7]
N   : 50 tokens  (41005014 → 42001001a)
OUT : ClCntfcl_openai_abl1_41005014_42001001a_20260326.jsonl

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl1  [batch 4/7]
N   : 50 tokens  (42001001b → 42008056)
OUT : ClCntfcl_openai_abl1_42001001b_42008056_20260326.jsonl

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl1  [batch 5/7]
N   : 50 tokens  (42009008a → 42019037)
OUT : ClCntfcl_openai_abl1_42009008a_42019037_20260326.jsonl

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl1  [batch 6/7]
N   : 50 tokens  (42019046 → 43005033)
OUT : ClCntfcl_openai_abl1_42019046_43005033_20260326.jsonl

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl1  [batch 7/7]
N 

/var/folders/22/4lgkjkkn2vvb2m8ffgsjl7gh0000gn/T/ipykernel_46633/856294939.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp      = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl2  [batch 2/7]
N   : 50 tokens  (40019028 → 41004029)
OUT : ClCntfcl_openai_abl2_40019028_41004029_20260326.jsonl

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl2  [batch 3/7]
N   : 50 tokens  (41005014 → 42001001a)
OUT : ClCntfcl_openai_abl2_41005014_42001001a_20260326.jsonl

Done: 49 processed, 1 parse failures

RUN : ClCntfcl | openai | abl2  [batch 4/7]
N   : 50 tokens  (42001001b → 42008056)
OUT : ClCntfcl_openai_abl2_42001001b_42008056_20260326.jsonl

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl2  [batch 5/7]
N   : 50 tokens  (42009008a → 42019037)
OUT : ClCntfcl_openai_abl2_42009008a_42019037_20260326.jsonl

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl2  [batch 6/7]
N   : 50 tokens  (42019046 → 43005033)
OUT : ClCntfcl_openai_abl2_42019046_43005033_20260326.jsonl

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl2  [batch 7/7]
N 

/var/folders/22/4lgkjkkn2vvb2m8ffgsjl7gh0000gn/T/ipykernel_46633/856294939.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp      = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl3  [batch 2/7]
N   : 50 tokens  (40019028 → 41004029)
OUT : ClCntfcl_openai_abl3_40019028_41004029_20260326.jsonl

Done: 50 processed, 0 parse failures

RUN : ClCntfcl | openai | abl3  [batch 3/7]
N   : 50 tokens  (41005014 → 42001001a)
OUT : ClCntfcl_openai_abl3_41005014_42001001a_20260326.jsonl

